In [194]:
import torch
from pyqcu import tools, dslash, lattice
from pyqcu.cuda import qcu, define
from pyqcu.cuda.define import params, argv, set_ptrs
params[define._LAT_X_] = 4
params[define._LAT_Y_] = 4
params[define._LAT_Z_] = 4
params[define._LAT_T_] = 8
params[define._LAT_XYZT_] = params[define._LAT_X_] * \
    params[define._LAT_Y_]*params[define._LAT_Z_]*params[define._LAT_T_]
params[define._GRID_X_], params[define._GRID_Y_], params[define._GRID_Z_], params[
    define._GRID_T_] = tools.give_grid_size()
params[define._PARITY_] = 0
params[define._NODE_RANK_] = define.rank
params[define._NODE_SIZE_] = define.size
params[define._DAGGER_] = 0
params[define._MAX_ITER_] = 1000
params[define._DATA_TYPE_] = define._LAT_C64_
# params[define._DATA_TYPE_] = define._LAT_C128_
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = 1
params[define._MG_X_] = 1
params[define._MG_Y_] = 1
params[define._MG_Z_] = 1
params[define._MG_T_] = 1
params[define._LAT_E_] = 24
params[define._VERBOSE_] = 1
params[define._SEED_] = 42
argv = argv.to(dtype=define.dtype(params[define._DATA_TYPE_]).to_real())
argv[define._MASS_] = 0.05
argv[define._TOL_] = 1e-9
argv[define._SIGMA_] = 0.1
print(params)
print(argv)
print(set_ptrs)
gauge_eo = torch.zeros(size=[2, 3, 3, 4]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                       params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                            params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.rand_like(fermion_in_eo)
fermion_out_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                             params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))


tensor([   4,    4,    4,    8,  512,    1,    1,    1,    1,    0,    0,    1,
           0, 1000,    2,    0,    1,    1,    1,    1,    1,   24,    1,   42,
           0], dtype=torch.int32)
tensor([5.0000e-02, 1.0000e-09, 1.0000e-01])
tensor([94710544708304, 94710544713616, 94710544727344, 94710547407280,
        94710547399152, 94710545174048, 94710545176288,              0,
                     0,              0])


In [195]:
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = -1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyGaussGaugeQcu(gauge_eo, set_ptrs, params)

print(lattice.check_su3(U=gauge_eo[0]))
print(lattice.check_su3(U=gauge_eo[1]))

set_ptr:0x562384a561d0
set_ptrs:0x562372111840
long long set_ptr:94710549275088
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :0
host_params[_SET_PLAN_]      :-1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
hos

In [196]:
_gauge_eo=gauge_eo.clone()
_fermion_in_eo=fermion_in_eo.clone()
_fermion_out_eo=fermion_out_eo.clone()

In [197]:
print('Difference:', tools.norm(gauge_eo-_gauge_eo)/tools.norm(_gauge_eo))
print('Difference:', tools.norm(fermion_in_eo-_fermion_in_eo)/tools.norm(_fermion_in_eo))

Difference: 0.0
Difference: 0.0


In [ ]:

print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonBistabCgQcu(
    fermion_out_eo, fermion_in_eo, gauge_eo, set_ptrs, params)

print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))


tensor([94710549275088, 94710544713616, 94710544727344, 94710547407280,
        94710547399152, 94710545174048, 94710545176288,              0,
                     0,              0])
set_ptr:0x562384904d60
set_ptrs:0x562372111840
long long set_ptr:94710547893600
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :1
host_params[_SET_PLAN_]      :1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]     

In [ ]:

clover_ee = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_ee_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
# gauge_eo = torch.ones_like(gauge_eo)*2
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_ee, clover_ee_inv, gauge_eo, set_ptrs, params)

print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_oo, clover_oo_inv, gauge_eo, set_ptrs, params)

print(set_ptrs)
fermion_out_eo = torch.zeros_like(fermion_out_eo)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverBistabCgQcu(fermion_out_eo, fermion_in_eo, gauge_eo,
                           clover_ee, clover_oo, clover_ee_inv, clover_oo_inv,  set_ptrs, params)

print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_clover_term = dslash.make_clover(
    U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8))
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)+dslash.give_clover(src=qcu_dest, clover_term=refer_clover_term)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))

set_ptr:0x562384a400e0
set_ptrs:0x562372111840
long long set_ptr:94710549184736
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :2
host_params[_SET_PLAN_]      :2
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [200]:

clover_eeoo = torch.zeros(size=[2, 4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                                params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_eeoo[0] = clover_ee
clover_eeoo[1] = clover_oo
qcu_clover_term = tools.poooxyzt2oooxyzt(
    input_array=clover_eeoo)
qcu_clover_term = dslash.cut_I(clover_term=qcu_clover_term)

qcu_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=qcu_clover_term)
refer_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=refer_clover_term)
print('qcu_clover_term:', qcu_clover_term.flatten()[:100])
print('refer_clover_term:', refer_clover_term.flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[0].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[0].flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[1].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[1].flatten()[:100])
print('Difference:', tools.norm(refer_clover_term -
      qcu_clover_term)/tools.norm(qcu_clover_term))


qcu_clover_term: tensor([-3.8316e-03+0.j, -1.8643e-03+0.j,  3.2039e-03+0.j,  5.3358e-04+0.j,
        -1.7998e-03+0.j, -5.4159e-03+0.j,  2.0691e-03+0.j,  4.6600e-03+0.j,
        -5.0867e-03+0.j, -3.2208e-03+0.j,  9.8002e-04+0.j,  1.9778e-03+0.j,
         5.4185e-03+0.j,  3.1829e-05+0.j, -6.9264e-03+0.j, -5.8151e-03+0.j,
        -1.9433e-03+0.j,  3.4924e-03+0.j, -8.2582e-04+0.j, -2.0717e-03+0.j,
         6.3369e-03+0.j, -3.9542e-04+0.j, -1.4666e-03+0.j,  3.9876e-04+0.j,
         2.7074e-03+0.j, -1.4642e-03+0.j,  6.1376e-03+0.j,  2.6786e-04+0.j,
        -2.5557e-03+0.j,  3.4189e-04+0.j,  2.9922e-05+0.j,  7.3220e-03+0.j,
        -8.8704e-04+0.j,  2.8539e-04+0.j,  2.8368e-03+0.j, -2.4422e-03+0.j,
         1.9132e-03+0.j,  3.7313e-05+0.j, -2.8713e-03+0.j,  8.3513e-03+0.j,
         2.1443e-03+0.j, -3.1715e-03+0.j,  2.2817e-04+0.j, -1.7223e-03+0.j,
         4.2026e-03+0.j,  2.7715e-03+0.j, -1.9099e-03+0.j,  2.6815e-03+0.j,
        -3.4442e-03+0.j,  5.8651e-05+0.j,  3.4399e-03+0.j,  3.8651e-03+

In [201]:
print('qcu_dest:', qcu_dest.flatten()[:100])

qcu_dest: tensor([-2423.9595-1083.5673j, -2740.7827-1040.0818j, -2464.8394-932.2519j,
        -2632.3357-988.2342j, -2472.7339-966.3528j, -2753.9441-1123.8995j,
        -2429.5549-931.8877j, -2707.9521-1007.3556j, -2569.3740-1116.4802j,
        -2365.6360-893.5055j, -2593.4001-896.0939j, -2308.1638-956.6935j,
        -2626.5217-1049.7234j, -2427.0840-924.4254j, -2681.7251-1008.2121j,
        -2252.2610-987.2477j, -2316.2639-984.1031j, -2614.6453-1015.6442j,
        -2239.6155-957.9497j, -2576.3770-1020.6694j, -2296.9595-853.0179j,
        -2484.8923-1050.1718j, -2284.7192-975.2632j, -2512.0513-921.1801j,
        -2519.8379-989.8132j, -2343.9746-1018.6166j, -2624.2917-976.7189j,
        -2309.8345-956.3925j, -2675.9585-918.2230j, -2362.6978-988.4626j,
        -2621.3479-901.6094j, -2286.4336-925.8221j, -2686.0859-1001.6516j,
        -2543.9790-808.8883j, -2605.3459-902.1789j, -2451.7634-890.8874j,
        -2668.1855-858.8670j, -2518.5608-807.4714j, -2571.6912-885.9753j,
        -2448.19

In [202]:
a = qcu_src-refer_src
print('a:', a.flatten()[:100])

a: tensor([-13.5569-17.0862j, 576.3298+34.5238j, -14.2573-16.5642j,
        597.0721+37.5132j, -15.3313-16.5518j, 565.5590+60.7307j,
        -12.7065-17.3781j, 540.9052+65.7718j, 472.9958+50.4243j,
        -13.5949-14.8726j, 494.2865+10.3187j, -14.7516-17.9052j,
        519.4745+21.0386j, -14.6662-16.7678j, 485.5976+36.7007j,
        -12.0052-16.8017j, -11.4736-17.5593j, 502.1010+48.4071j,
        -14.8305-18.3796j, 503.3982+38.8152j, -13.8277-16.0479j,
        489.3999+21.7775j, -13.0205-15.8238j, 443.3875+30.0042j,
        440.9169+31.8121j, -13.9828-15.9407j, 489.2122+19.9442j,
        -14.0343-17.1197j, 482.4648+15.2021j, -12.6020-16.6315j,
        470.9854+29.5693j, -12.9244-17.8840j, 250.7564+139.2240j,
          4.7698-6.3322j, 290.2850+145.5429j,   4.3700-6.3555j,
        265.7173+107.3279j,   4.6290-6.8777j, 248.7515+136.6895j,
          6.1037-7.1453j,   4.9693-6.4792j, 165.0872+105.0316j,
          3.9614-6.8829j, 190.6026+104.6429j,   4.2737-6.9673j,
        182.8958+93.912

In [203]:
qcu_src

tensor([[[[[[3.2096e-03+8.7144e-01j, 9.6303e-01+2.4353e-01j,
             9.5490e-01+3.5100e-01j,  ...,
             9.1199e-01+6.2847e-01j, 5.8874e-01+8.7050e-01j,
             4.5500e-01+9.4897e-01j],
            [1.3655e-01+8.2710e-01j, 9.7308e-01+3.1054e-01j,
             9.1987e-01+5.2451e-01j,  ...,
             7.1724e-01+5.3741e-01j, 1.1629e-01+7.1132e-01j,
             1.9086e-01+9.4737e-01j],
            [8.2670e-01+9.5148e-01j, 5.7146e-01+8.9306e-01j,
             3.4530e-01+3.2319e-01j,  ...,
             3.7558e-01+1.9698e-01j, 1.6765e-01+2.8545e-01j,
             9.5233e-01+9.6440e-01j],
            [4.3458e-02+8.6024e-02j, 6.9263e-01+1.1442e-01j,
             6.6527e-01+8.2183e-01j,  ...,
             4.7057e-01+6.9802e-01j, 6.1698e-01+4.9709e-01j,
             7.4701e-01+7.7506e-01j]],

           [[4.5480e-01+4.7166e-01j, 5.8351e-02+3.9659e-02j,
             5.6039e-01+7.0716e-01j,  ...,
             6.5550e-01+2.1337e-01j, 1.6185e-01+3.6751e-01j,
             5.0548e-

In [204]:
refer_src

tensor([[[[[[ 1.3560e+01+1.7958e+01j, -5.7537e+02-3.4280e+01j,
              1.5212e+01+1.6915e+01j,  ...,
             -5.6465e+02-6.0102e+01j,  1.3295e+01+1.8249e+01j,
             -5.4045e+02-6.4823e+01j],
            [-4.7286e+02-4.9597e+01j,  1.4568e+01+1.5183e+01j,
             -4.9337e+02-9.7942e+00j,  ...,
              1.5383e+01+1.7305e+01j, -4.8548e+02-3.5989e+01j,
              1.2196e+01+1.7749e+01j],
            [ 1.2300e+01+1.8511e+01j, -5.0153e+02-4.7514e+01j,
              1.5176e+01+1.8703e+01j,  ...,
             -4.8902e+02-2.1580e+01j,  1.3188e+01+1.6109e+01j,
             -4.4244e+02-2.9040e+01j],
            [-4.4087e+02-3.1726e+01j,  1.4675e+01+1.6055e+01j,
             -4.8855e+02-1.9122e+01j,  ...,
              1.3073e+01+1.7329e+01j, -4.7037e+02-2.9072e+01j,
              1.3671e+01+1.8659e+01j]],

           [[-2.5030e+02-1.3875e+02j, -4.7114e+00+6.3719e+00j,
             -2.8972e+02-1.4484e+02j,  ...,
             -3.9735e+00+7.0911e+00j, -2.4859e+02-1.363

In [ ]:
fermion_out_eo = torch.zeros_like(fermion_out_eo)
fermion_in_o=fermion_in_eo[1]
fermion_out_e=fermion_out_eo[0]
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverDslashQcu(fermion_out_e, fermion_in_o, gauge_eo, set_ptrs, params)

refer_clover_term = dslash.make_clover(
    U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8))
refer_clover_term = dslash.add_I(clover_term=refer_clover_term)
refer_clover_term_inv = dslash.inverse(clover_term=refer_clover_term)
refer_clover_term_inv_eo = tools.oooxyzt2poooxyzt(
    input_array=refer_clover_term_inv)
refer_clover_term_inv_e = refer_clover_term_inv_eo[0]
refer_fermion_out_e = dslash.give_clover_ee(src_e=dslash.give_wilson_eo(
    src_o=fermion_in_o, U_eo=gauge_eo, kappa=1 / (2 * argv[define._MASS_] + 8)),clover_e=refer_clover_term_inv_e)
print(tools.norm(clover_ee_inv-refer_clover_term_inv_e)/tools.norm(clover_ee_inv))
print(tools.norm(fermion_out_e-refer_fermion_out_e)/tools.norm(fermion_out_e))

set_ptr:0x5623847d3550
set_ptrs:0x562372111840
long long set_ptr:94710546642256
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :1
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :5
host_params[_SET_PLAN_]      :2
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [206]:
fermion_out_eo = torch.zeros_like(fermion_out_eo)
fermion_in_o=fermion_in_eo[1]
fermion_out_e=fermion_out_eo[0]
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonDslashQcu(fermion_out_e, fermion_in_o, gauge_eo, set_ptrs, params)

refer_fermion_out_e = dslash.give_wilson_eo(
    src_o=fermion_in_o, U_eo=gauge_eo, kappa=1 / (2 * argv[define._MASS_] + 8))
print(fermion_out_e.flatten()[:100])
print(refer_fermion_out_e.flatten()[:100])
print(tools.norm(fermion_out_e-refer_fermion_out_e)/tools.norm(fermion_out_e))

set_ptr:0x56238490c2a0
set_ptrs:0x562372111840
long long set_ptr:94710547923616
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :1
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :6
host_params[_SET_PLAN_]      :1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [207]:
print("gauge_eo.is_contiguous():",gauge_eo.is_contiguous())
print("fermion_in_eo.is_contiguous():",fermion_in_eo.is_contiguous())
print("fermion_in_out.is_contiguous():",fermion_out_eo.is_contiguous())
print("qcu_src.is_contiguous():",qcu_src.is_contiguous())
print("qcu_dest.is_contiguous():",qcu_dest.is_contiguous())

gauge_eo.is_contiguous(): True
fermion_in_eo.is_contiguous(): True
fermion_in_out.is_contiguous(): True
qcu_src.is_contiguous(): True
qcu_dest.is_contiguous(): True
